# 01 — Exploratory Data Analysis (EDA)

**Project:** Cinema Ticket Demand Forecasting (`T-48h`)  
**Purpose:** Inspect raw data quality (missingness, duplicates, outliers), target distributions, demand drivers (weekday, slot, genre, movie age), capacity censoring (`sold_out`), correlations, and time-series behaviour. All figures are saved to `reports/figures/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config

sns.set_theme(style="whitegrid", palette="deep")
cfg = load_config(PROJECT_ROOT / "config.yaml")
FIG_DIR = PROJECT_ROOT / cfg["paths"]["figures_dir"]
FIG_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(PROJECT_ROOT / cfg["paths"]["raw_data"])
clean_df = pd.read_parquet(PROJECT_ROOT / cfg["paths"]["cleaned_data"])

print(f"Raw rows: {len(raw_df):,} | Clean rows: {len(clean_df):,}")
print(f"Date range: {clean_df['show_date'].min()} to {clean_df['show_date'].max()}")
print(f"Movies: {clean_df['movie_id'].nunique()} | Screens: {clean_df['screen_id'].nunique()}")
print(f"Sold-out shows: {clean_df['sold_out'].sum():,} ({clean_df['sold_out'].mean()*100:.2f}%)")

Raw rows: 21,708 | Clean rows: 21,584
Date range: 2024-01-01 to 2025-06-23
Movies: 65 | Screens: 8
Sold-out shows: 304 (1.41%)


## 1. Data Quality Audit (Raw Dataset)
Check missing values, exact duplicates, and out-of-range POS anomalies before cleaning.

In [2]:
missing_counts = raw_df.isna().sum()
missing_pct = (missing_counts / len(raw_df) * 100).round(3)
quality_summary = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
print("=== Missingness in Raw Data ===")
print(quality_summary[quality_summary["missing_count"] > 0])

dup_count = raw_df.duplicated(subset=["show_id"]).sum()
bad_price = ((raw_df["ticket_price"] < 4.0) | (raw_df["ticket_price"] > 60.0)).sum()
bad_runtime = ((raw_df["runtime_min"] < 60) | (raw_df["runtime_min"] > 240)).sum()
neg_tickets = (raw_df["tickets_sold"] < 0).sum()
over_cap = (raw_df["tickets_sold"] > raw_df["capacity"]).sum()

print(f"\nDuplicate show_ids: {dup_count}")
print(f"Price outliers (<4 or >60): {bad_price}")
print(f"Runtime outliers (<60 or >240): {bad_runtime}")
print(f"Negative tickets_sold: {neg_tickets} | Over-capacity tickets_sold: {over_cap}")

=== Missingness in Raw Data ===
              missing_count  missing_pct
language                144        0.663
rating                  135        0.622
runtime_min             132        0.608
ticket_price            127        0.585

Duplicate show_ids: 108
Price outliers (<4 or >60): 16
Runtime outliers (<60 or >240): 16
Negative tickets_sold: 16 | Over-capacity tickets_sold: 16


## 2. Target Distributions (`tickets_sold` and `occupancy_rate`)

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(clean_df["tickets_sold"], bins=35, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Distribution of Tickets Sold")
axes[0].set_xlabel("Tickets Sold")

sns.histplot(clean_df["occupancy_rate"], bins=30, kde=False, ax=axes[1], color="#d95f02")
axes[1].axvline(1.0, color="red", linestyle="--", label=f"Capacity Cap (Sold Out: {clean_df['sold_out'].mean()*100:.1f}%)")
axes[1].set_title("Distribution of Seat Occupancy Rate (Censored at 1.0)")
axes[1].set_xlabel("Occupancy Rate (tickets_sold / capacity)")
axes[1].legend()

plt.tight_layout()
fig.savefig(FIG_DIR / "eda_01_distributions.png", dpi=150)
plt.show()
print(clean_df[["tickets_sold", "occupancy_rate", "capacity", "ticket_price"]].describe().round(2))

       tickets_sold  occupancy_rate  capacity  ticket_price
count      21584.00        21584.00  21584.00      21584.00
mean          47.23            0.27    170.51         14.48
std           46.38            0.22     69.28          3.73
min            2.00            0.02     85.00          6.00
25%           16.00            0.11    118.00         11.77
50%           33.00            0.21    174.00         13.91
75%           63.00            0.38    239.00         16.88
max          302.00            1.00    302.00         24.80


/var/folders/tb/djbkr2zj7x5c69pj0wqhqhzh0000gn/T/ipykernel_38159/366958607.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Key Demand Drivers: Weekday, Slot, Genre, and Movie Age

In [4]:
eda_df = clean_df.copy()
eda_df["weekday"] = eda_df["show_datetime"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

eda_df["movie_age_days"] = (eda_df["show_datetime"].dt.normalize() - eda_df["release_date"]).dt.days
eda_df["age_bucket"] = pd.cut(
    eda_df["movie_age_days"],
    bins=[-1, 3, 7, 14, 30, 1000],
    labels=["0-3d (Opening)", "4-7d", "8-14d", "15-30d", "31+d"],
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(data=eda_df, x="weekday", y="occupancy_rate", order=weekday_order, ax=axes[0, 0], errorbar="ci")
axes[0, 0].set_title("Mean Occupancy by Day of Week")
axes[0, 0].tick_params(axis="x", rotation=25)

sns.barplot(data=eda_df, x="slot", y="occupancy_rate", order=["matinee", "evening", "late"], ax=axes[0, 1], errorbar="ci")
axes[0, 1].set_title("Mean Occupancy by Showtime Slot")

genre_order = eda_df.groupby("genre")["occupancy_rate"].mean().sort_values(ascending=False).index
sns.barplot(data=eda_df, x="genre", y="occupancy_rate", order=genre_order, ax=axes[1, 0], errorbar="ci")
axes[1, 0].set_title("Mean Occupancy by Genre")
axes[1, 0].tick_params(axis="x", rotation=25)

sns.barplot(data=eda_df, x="age_bucket", y="occupancy_rate", ax=axes[1, 1], errorbar="ci")
axes[1, 1].set_title("Mean Occupancy by Movie Age (Decay Curve)")

plt.tight_layout()
fig.savefig(FIG_DIR / "eda_02_demand_drivers.png", dpi=150)
plt.show()

/var/folders/tb/djbkr2zj7x5c69pj0wqhqhzh0000gn/T/ipykernel_38159/857794869.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Occupancy vs Capacity & Sold-Out Share

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

screen_stats = clean_df.groupby(["screen_id", "screen_type", "capacity"], as_index=False).agg(
    mean_occ=("occupancy_rate", "mean"),
    sold_out_pct=("sold_out", lambda s: s.mean() * 100),
).sort_values("capacity")

sns.scatterplot(data=screen_stats, x="capacity", y="mean_occ", hue="screen_type", s=120, ax=axes[0])
axes[0].set_title("Mean Occupancy Rate vs Screen Capacity")
axes[0].set_ylabel("Mean Occupancy Rate")

sns.barplot(data=screen_stats, x="screen_id", y="sold_out_pct", hue="screen_type", dodge=False, ax=axes[1])
axes[1].set_title("Sold-Out Share (%) by Screen")
axes[1].set_ylabel("Sold-Out Shows (%)")

plt.tight_layout()
fig.savefig(FIG_DIR / "eda_03_capacity_and_censoring.png", dpi=150)
plt.show()
print(screen_stats.round(3).to_string(index=False))

screen_id screen_type  capacity  mean_occ  sold_out_pct
   SCR-01    Standard        85     0.265         1.187
   SCR-07         VIP       100     0.274         1.297
   SCR-02    Standard       118     0.259         1.038
   SCR-03    Standard       147     0.261         1.112
   SCR-04       Dolby       174     0.277         1.297
   SCR-05    Standard       199     0.266         1.112
   SCR-06       Dolby       239     0.284         1.853
   SCR-08        IMAX       302     0.298         2.372


/var/folders/tb/djbkr2zj7x5c69pj0wqhqhzh0000gn/T/ipykernel_38159/4285372454.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Missingness, Outliers & Numeric Correlations

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

miss_plot = quality_summary[quality_summary["missing_count"] > 0]
sns.barplot(x=miss_plot.index, y=miss_plot["missing_pct"], ax=axes[0], color="#7570b3")
axes[0].set_title("Raw Data Missingness (%)")
axes[0].set_ylabel("Missing (%)")

num_cols = ["capacity", "runtime_min", "ticket_price", "is_holiday", "is_special_event", "tickets_sold", "occupancy_rate", "sold_out"]
corr = eda_df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1], cbar=False)
axes[1].set_title("Feature & Target Correlation Matrix")

plt.tight_layout()
fig.savefig(FIG_DIR / "eda_04_quality_and_corr.png", dpi=150)
plt.show()

/var/folders/tb/djbkr2zj7x5c69pj0wqhqhzh0000gn/T/ipykernel_38159/1546301053.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Time-Series View (Daily Ticket Sales & Rolling Occupancy)

In [7]:
daily = clean_df.groupby("show_date", as_index=False).agg(
    total_tickets=("tickets_sold", "sum"),
    mean_occ=("occupancy_rate", "mean"),
    is_holiday=("is_holiday", "max"),
)
daily["show_date"] = pd.to_datetime(daily["show_date"])
daily = daily.sort_values("show_date")
daily["occ_7d_ma"] = daily["mean_occ"].rolling(7, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(daily["show_date"], daily["total_tickets"], alpha=0.55, label="Daily Tickets Sold")
axes[0].plot(daily["show_date"], daily["total_tickets"].rolling(7).mean(), color="#d95f02", linewidth=2, label="7-Day Moving Avg")
axes[0].set_title("Daily Cinema Ticket Volume (18 Months)")
axes[0].set_ylabel("Total Tickets")
axes[0].legend()

axes[1].plot(daily["show_date"], daily["mean_occ"], alpha=0.45, label="Daily Mean Occupancy")
axes[1].plot(daily["show_date"], daily["occ_7d_ma"], color="#1b9e77", linewidth=2, label="7-Day Moving Avg")
hol = daily[daily["is_holiday"] == 1]
axes[1].scatter(hol["show_date"], hol["mean_occ"], color="red", s=25, zorder=5, label="Public Holiday")
axes[1].set_title("Daily Mean Occupancy Rate & Holiday Spikes")
axes[1].set_ylabel("Occupancy Rate")
axes[1].legend()

plt.tight_layout()
fig.savefig(FIG_DIR / "eda_05_timeseries.png", dpi=150)
plt.show()

/var/folders/tb/djbkr2zj7x5c69pj0wqhqhzh0000gn/T/ipykernel_38159/1641462558.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
